# BóSight — Week 6: Body Condition Score (BCS)

**Owner:** Person 2

## Important context
MmCows contains **no BCS ground truth** — no vet body-condition scores, no weights.
This is documented in the data guide (Section 16). Therefore Week 6 is **not** a
trained/validated model. Instead we build a **transparent geometric proxy** so the
downstream pipeline (Week 7 fusion, Week 9 dashboard) has the required `bcs_daily.parquet`
output, and we clearly label it as a proxy in the report.

## Proxy definition
- A cow's **apparent body size** in the image correlates loosely with body condition
  (larger frame-fill = more mass/condition).
- We use the **bounding-box area** from the tracking output (`tracked_cows.parquet`).
- We restrict to **standing frames only** (using the behaviour ground-truth labels) so
  that lying posture — which drastically changes bbox shape — doesn't confound the estimate.
- We take the **median** bbox area per cow (robust to outliers), then **min-max scale**
  across the herd into a conservative **2.0–4.0** BCS band (we avoid the extreme 1.0 / 5.0
  ends, which would not be justified by a proxy).

## Limitations (state these in the report)
- Isometric camera view: cows nearer/farther fill different areas → perspective confound.
- No ground truth → cannot compute MAE/R². This is a **relative** ranking, not calibrated BCS.
- In a real deployment, a dedicated BCS camera (rump view) + vet scores would replace this.

**Inputs to attach on Kaggle:** the tracking output (`tracked_cows.parquet`) and the
behaviour labels (16 `C0n_0725.csv`). Notebook auto-detects paths.

## 1. Setup & auto-find inputs

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# --- find tracked_cows.parquet ---
hits = glob.glob('/kaggle/input/**/tracked_cows.parquet', recursive=True)
assert hits, 'tracked_cows.parquet not found - attach the tracking dataset.'
TRACK_PATH = hits[0]; print('tracking:', TRACK_PATH)

# --- find behaviour labels dir (has C01_0725.csv) ---
BEH_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('C01_0725') and f.endswith('.csv') for f in files):
        BEH_DIR = root; break
assert BEH_DIR, 'behaviour labels not found - attach the behaviour-labels dataset.'
print('beh labels:', BEH_DIR)

OUT_DIR = '/kaggle/working'
DATE = '2023-07-25'
STANDING_CODE = 2   # MmCows behaviour code for 'standing'

## 2. Confirm there is no BCS ground truth (documentation step)

In [ ]:
# Search all attached inputs for anything that looks like BCS / body condition / weight.
found = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        low = f.lower()
        if any(k in low for k in ['bcs', 'body_condition', 'bodycondition', 'weight', 'condition_score']):
            found.append(os.path.join(root, f))
print('BCS-like files found:', found if found else 'NONE')
print('\nConfirmed: MmCows provides no BCS ground truth -> using a documented proxy.')

## 3. Load tracking + behaviour, keep standing detections only

In [ ]:
track = pd.read_parquet(TRACK_PATH)
print('tracking rows:', len(track), '| columns:', list(track.columns))

# unix timestamp from the 'stem' column ('{unix_ts}_{HH-MM-SS}')
track['ts'] = track['stem'].str.split('_').str[0].astype(int)

# bbox area in pixels
track['bbox_area'] = (track['x2'] - track['x1']).clip(lower=0) * (track['y2'] - track['y1']).clip(lower=0)

# behaviour lookup: (cow, ts) -> code
beh = {}
for c in range(1, 17):
    cow = f'C{c:02d}'
    df = pd.read_csv(os.path.join(BEH_DIR, f'{cow}_0725.csv'))
    beh[cow] = dict(zip(df['timestamp'].astype(int), df['behavior'].astype(int)))

def code_for(row):
    return beh.get(row['cow_id'], {}).get(row['ts'], -1)

track['beh_code'] = track.apply(code_for, axis=1)
standing = track[track['beh_code'] == STANDING_CODE].copy()
print('standing detections:', len(standing), 'of', len(track))

## 4. Per-cow proxy: median standing bbox area → scaled BCS (2.0–4.0)

In [ ]:
g = standing.groupby('cow_id')['bbox_area'].median().rename('median_area')
prox = g.reset_index()

# min-max scale median area across the herd into [2.0, 4.0]
lo, hi = prox['median_area'].min(), prox['median_area'].max()
prox['bcs_estimate'] = 2.0 + (prox['median_area'] - lo) / (hi - lo) * (4.0 - 2.0)
prox['bcs_estimate'] = prox['bcs_estimate'].round(2)

# n_observations = standing detections used per cow
counts = standing.groupby('cow_id').size().rename('n_observations')
prox = prox.merge(counts, on='cow_id')
prox['date'] = DATE

bcs_daily = prox[['cow_id', 'date', 'bcs_estimate', 'n_observations']].sort_values('cow_id').reset_index(drop=True)
print(bcs_daily.to_string())

## 5. Visualise the relative ranking

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
d = bcs_daily.sort_values('bcs_estimate')
ax.barh(d['cow_id'], d['bcs_estimate'], color='#55A868')
ax.set_xlabel('BCS estimate (proxy, 2.0–4.0)'); ax.set_title('Relative body-condition proxy by cow')
ax.axvline(3.0, color='grey', ls='--', lw=1, label='mid (3.0)')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Save output (schema = two_person_plan §2.4)

In [ ]:
out = os.path.join(OUT_DIR, 'bcs_daily.parquet')
bcs_daily.to_parquet(out, index=False)

# also save a small note documenting the method + limitations
import json
meta = {
    'method': 'geometric proxy — median standing-frame bbox area, min-max scaled to 2.0-4.0',
    'ground_truth_available': False,
    'range': [2.0, 4.0],
    'confounds': ['isometric camera perspective', 'no calibration', 'relative ranking only'],
    'n_cows': int(len(bcs_daily)),
}
with open(os.path.join(OUT_DIR, 'bcs_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('SAVED:', out)
print('SAVED:', os.path.join(OUT_DIR, 'bcs_meta.json'))
print('\nDownload both from the Output tab (or Save Version) and copy to D:/D2/bosight/outputs/')

## Next steps
- **Download** `bcs_daily.parquet` + `bcs_meta.json` and save to `D:/D2/bosight/outputs/`.
- In the report, present this as a **relative body-condition proxy**, not a calibrated BCS,
  and cite the absence of ground truth as the reason.
- **Week 7 (fusion):** this proxy becomes one feature per cow alongside behaviour budget,
  IMU, CBT, and UWB statistics.